<h1 style="color:red">Use of sklearn pipelines </h1>

In [1]:
import pandas as pd 
import numpy as np

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.tree import DecisionTreeClassifier

<img src='pipeline_flow.png'>

In [3]:
df = pd.read_csv("titanic.csv") 
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [6]:
df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'], inplace=True)
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [34]:
# Train test split
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df['Survived'], test_size=0.3, random_state=0) 

X_train.shape, X_test.shape 

((623, 7), (268, 7))

In [8]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
857,1,male,51.0,0,0,26.5500,S
52,1,female,49.0,1,0,76.7292,C
386,3,male,1.0,5,2,46.9000,S
124,1,male,54.0,0,1,77.2875,S
578,3,female,NaN,1,0,14.4583,C


In [17]:
# imputation transformer 
tnf1 = ColumnTransformer(transformers=[ 
                        ('impute_age', SimpleImputer(), [2]), 
                        ('impute_embarked', SimpleImputer(strategy='most_frequent'), [-1])
                       ],
                    remainder='passthrough'
)
# output will be in numpy array, so better to use indexes, *no New numpy array* is created, it modifies the original numpy array. 
#  if you convert them to dataframes first, then better to use column names.

In [25]:
# OHE transformer 
tnf2 = ColumnTransformer(transformers=[
                          ('ohe_sex_embarked', OneHotEncoder(sparse_output=False, dtype=int, handle_unknown='ignore'), [1,6])
                         ],
                         remainder='passthrough'
                        )     # we are not dropping the first column

In [29]:
tnf2

ColumnTransformer(remainder='passthrough',
                  transformers=[('ohe_sex_embarked',
                                 OneHotEncoder(dtype=<class 'int'>,
                                               handle_unknown='ignore',
                                               sparse_output=False),
                                 [1, 6])])

In [27]:
# Scaling 
# while using Feature selection, we use MinMaxScaler
tnf3 = ColumnTransformer(transformers=[('scale', MinMaxScaler(), slice(0,10) )])       # Applies MinMaxScaler in columns 0 to 9 all

In [28]:
tnf3

ColumnTransformer(transformers=[('scale', MinMaxScaler(), slice(0, 10, None))])

In [32]:
# Feature Selection
tnf4 = SelectKBest(score_func=chi2, k=8)    

# chi2 = Chi-Square statistical test ( Used to measure dependency between feature and target )
 # Interpretation:     Higher chi-square score → feature is more relevant
# k=8, Selects the top 8 features

In [33]:
# train the model
tnf5 = DecisionTreeClassifier()

<h3 style="color:brown">Create Pipeline</h3>

In [56]:
pipe = Pipeline([
    ('tnf1', tnf1),
    ('tnf2', tnf2),
    ('tnf3', tnf3),
    ('tnf4', tnf4),
    ('tnf5', tnf5)
])
# passing a list of tuple in which 2 things are passed, transformer name and object.

<h3 style="color:brown">Pipeline Vs make_pipeline</h3>

In [57]:
# Pipeline requires naming of steps, make_pipeline does not.

# (Same applies to ColumnTransformer vs make_column_transformer)

In [58]:
# Alternate Syntax 
# pipe = make_pipeline(tnf1, tnf2, tnf3, tnf4, tnf5)

In [59]:
# train
pipe.fit(X_train, y_train)        # we are calling .fit() as we are training the model also, so that we can call predict later onn.

# if you are not training the model, then you have to call fit_transform

# if you pipeline consists of :   impute  -->  OHE  -->  Scaling  
# call      pipe.fit_transform(X_train)

Pipeline(steps=[('tnf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [-1])])),
                ('tnf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_embarked',
                                                  OneHotEncoder(dtype=<class 'int'>,
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 6])])),
                ('tnf3',
                 ColumnTransformer(transformers=[('scale', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('tnf4',
                 SelectKBest(k=8,
                             score_func=<function chi2 at 0x00000147D97F9DA0>)),
                ('tnf5', DecisionTreeClassifier())])

<h3 style="color:brown">Explore Pipeline</h3>

In [60]:
# pipe object is trained now
pipe.named_steps        # it give all the steps that this pipeline is following

{'tnf1': ColumnTransformer(remainder='passthrough',
                   transformers=[('impute_age', SimpleImputer(), [2]),
                                 ('impute_embarked',
                                  SimpleImputer(strategy='most_frequent'),
                                  [-1])]),
 'tnf2': ColumnTransformer(remainder='passthrough',
                   transformers=[('ohe_sex_embarked',
                                  OneHotEncoder(dtype=<class 'int'>,
                                                handle_unknown='ignore',
                                                sparse_output=False),
                                  [1, 6])]),
 'tnf3': ColumnTransformer(transformers=[('scale', MinMaxScaler(), slice(0, 10, None))]),
 'tnf4': SelectKBest(k=8, score_func=<function chi2 at 0x00000147D97F9DA0>),
 'tnf5': DecisionTreeClassifier()}

In [66]:
# pipe.named_steps['tnf1']          # fetcing details of 'tnf1'
# pipe.named_steps['tnf1'].transformers_     # gives a list of all transformers present in 'tnf1'
# pipe.named_steps['tnf1'].transformers_[0]     # accessing the first transformer of 'tnf1'                             (accessing list 0 index)
# pipe.named_steps['tnf1'].transformers_[0][1]     # accessing the SimpleImputer() of first transformer of 'tnf1'       (accessing tuple 1 index)
pipe.named_steps['tnf1'].transformers_[0][1].statistics_    # giving the mean value of SimpleImputer() of first transformer of 'tnf1'

array([29.91533865])

In [74]:
pipe.named_steps['tnf1'].transformers_[1][1].statistics_        
# fetching the statistics of impute_embarked (most frequent value)

array(['S'], dtype=object)

In [83]:
#  Predict 
y_pred = pipe.predict(X_test)

y_pred

array([1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0,
       0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0,
       1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0,
       0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1,
       0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0,
       0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0,
       0, 1, 0, 1])

In [84]:
from sklearn.metrics import accuracy_score
accuracy_score(y_pred, y_test)          # by feature selection we have eliminated some columns, so accuracy is a bit low.

0.6417910447761194

<h3 style="color:brown">Cross Validation using pipeline</h3>

In [87]:
# Why do we use cross-validation?
    # Reduces overfitting
    # Gives stable and unbiased performance estimate
    # Better than single train-test split

# cross validation using cross_val_score
from sklearn.model_selection import cross_val_score

cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy').mean()

# cv=5,   Perform 5-fold cross-validation on the training data
# - X_train : input features (training data)
# - y_train : target labels (training data)
# - cv=5    : split data into 5 folds      (train on 4 folds, validate on 1 fold, repeat 5 times)
# - scoring='accuracy' : use accuracy as the evaluation metric

np.float64(0.6324387096774193)

<h3 style="color:brown">GridSearch using pipeline</h3>

In [97]:
# Hyper-parameter tuning :- you can change the setting to improve it's performance. 

# gridsearchcv
params = {
    'tnf5__max_depth' : [1,2,3,4,5,None]
}   
# max_depth parameter of decisionTree, changing this params will improve/reduce it's performance.
# you have to mention the name of the model with 2 underscore,  e.g :-  tnf5__

In [98]:
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(pipe, params, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('tnf1',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('impute_age',
                                                                         SimpleImputer(),
                                                                         [2]),
                                                                        ('impute_embarked',
                                                                         SimpleImputer(strategy='most_frequent'),
                                                                         [-1])])),
                                       ('tnf2',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('ohe_sex_embarked',
                                                                         OneHotEncoder(dtype=<class 'int'>,
                                                                                       handle_unknown='ignore',
                                                                                       sparse_output=False),
                                                                         [1,
                                                                          6])])),
                                       ('tnf3',
                                        ColumnTransformer(transformers=[('scale',
                                                                         MinMaxScaler(),
                                                                         slice(0, 10, None))])),
                                       ('tnf4',
                                        SelectKBest(k=8,
                                                    score_func=<function chi2 at 0x00000147D97F9DA0>)),
                                       ('tnf5', DecisionTreeClassifier())]),
             param_grid={'tnf5__max_depth': [1, 2, 3, 4, 5, None]},
             scoring='accuracy')

In [99]:
grid.best_score_

np.float64(0.6324387096774193)

In [100]:
grid.best_params_

{'tnf5__max_depth': 5}

<h3 style="color:brown">Exporting the pipeline</h3>

In [102]:
import pickle

In [105]:
pickle.dump(pipe, open('pipeline_model/pipe.pkl','wb'))      # export the pipeline object to the given destination